# Analiza datasetu


In [77]:
from pathlib import Path
import pandas as pd

## Wczytywanie danych

In [78]:
base_path = Path("..") / "data"
data_path = base_path / "data_raw"

spam_df = pd.read_csv(
    data_path,
    sep="\t",
    header=None,
    names=["label", "text"],
)

spam_df.head()

,label,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


## Analiza struktury danych i unikatowych wartości

In [79]:
spam_df.shape

(5572, 2)

In [80]:
spam_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   label   5572 non-null   str  
 1   text    5572 non-null   str  
dtypes: str(2)
memory usage: 87.2 KB


In [81]:
spam_df.describe(include="all")

,label,text
count,5572,5572
unique,2,5169
top,ham,"Sorry, I'll call later"
freq,4825,30


In [82]:
spam_df["label"].unique()

<StringArray>
['ham', 'spam']
Length: 2, dtype: str

In [83]:
(spam_df["text"].str.strip() == "").sum()

np.int64(0)

> Po wstępnym przeanalizowaniu datasetu, możemy zauważyć, że `label` ma tylko 2 wartości (jak i oczekiwaliśmy), natomiast wartości w kolumnie `text` powtarzają się.

> Także możemy zauważyć, że brak jest wartości pustych typu `NaN` lub `wiadomości składających się z białych znaków`.

## Analiza duplikatów(jednakowych wiadomości)

In [84]:
spam_df.duplicated().sum()

np.int64(403)

In [85]:
spam_df[spam_df.duplicated(keep=False)].sort_values(by="text")

,label,text
505,spam,+123 Congratulations - in this week's competit...
2124,spam,+123 Congratulations - in this week's competit...
2344,ham,1) Go to write msg 2) Put on Dictionary mode 3...
1373,ham,1) Go to write msg 2) Put on Dictionary mode 3...
2163,ham,1) Go to write msg 2) Put on Dictionary mode 3...
...,...,...
1381,ham,i dnt wnt to tlk wid u
1412,ham,somewhere out there beneath the pale moon ligh...
4004,ham,somewhere out there beneath the pale moon ligh...
2389,ham,wiskey Brandy Rum Gin Beer Vodka Scotch Shampa...


> Widzimy, że w zbiorze występują duplikaty. Usuwamy je, pozostawiając pierwsze wystąpienie każdej wiadomości.

In [86]:
spam_df = spam_df.drop_duplicates()
spam_df.duplicated().sum()

np.int64(0)

In [87]:
spam_df.info()

<class 'pandas.DataFrame'>
Index: 5169 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   label   5169 non-null   str  
 1   text    5169 non-null   str  
dtypes: str(2)
memory usage: 121.1 KB


In [88]:
spam_df.describe(include="all")

,label,text
count,5169,5169
unique,2,5169
top,ham,"Go until jurong point, crazy.. Available only ..."
freq,4516,1


> Opis `dataframe` pokazuje, że usuwanie duplikatów zostało wykonane pomyślnie

## Sprawdzenie konfliktu etykiet

> Warto również sprawdzić, czy nie występują wiadomości, które są jednocześnie oznaczone jako `spam` i `ham`, ponieważ taki konflikt etykiet jest niepożądany

In [89]:
conflicting_labels = (
    spam_df.groupby("text")["label"]
    .nunique()
)
conflicting_labels[conflicting_labels > 1]

Series([], Name: label, dtype: int64)

> W zbiorze nie występują konflikty etykiet

## Mapowanie wartości kategoryjnych

In [90]:
spam_df["label"] = spam_df.label.map({'spam': 1, "ham": 0})

## Zapisywanie przygotowanych danych

In [91]:
output_path = base_path / "data_processed.csv"
spam_df.to_csv(output_path, index=False)

## Podsumowanie

> Dane zostały wyczyszczone i przygotowane do następnych etapów pracy